# ELECTRE in depth

The [Quick Start](quickstart.ipynb) tutorial used `ELECTRE1` briefly to
find a "kernel" of good alternatives. This tutorial goes deeper into the
`ELECTRE` family: how concordance/discordance actually work, how
`ELECTRE2` differs from `ELECTRE1`, why their output should be read
differently from a plain ranking, how sensitive the kernel is to the
threshold parameters, and when this whole family is a better fit than a
compensatory method like `TOPSIS` or `WeightedSumModel`.

## Case

We reuse the business-site-selection matrix from
<cite data-cite="cebrian2009localizacion">[cebrian2009localizacion]</cite>,
an empirical application of `ELECTRE I` to six candidate business
locations in Aragón, Spain, evaluated on five criteria: four to maximize
(*C1*, *C2*, *C4*, *C5*) and one to minimize (*C3*, an investment-risk
indicator).

In [1]:
import skcriteria as skc
from skcriteria.preprocessing.scalers import SumScaler

dm = skc.mkdm(
    matrix=[
        [6, 5, 28, 5, 5],
        [4, 2, 25, 10, 9],
        [5, 7, 35, 9, 6],
        [6, 1, 27, 6, 7],
        [6, 8, 30, 7, 9],
        [5, 6, 26, 4, 8],
    ],
    objectives=[max, max, min, max, max],
    weights=[0.25, 0.25, 0.10, 0.20, 0.20],
    alternatives=["L1", "L2", "L3", "L4", "L5", "L6"],
    criteria=["C1", "C2", "C3", "C4", "C5"],
)

# ELECTRE needs every criterion on a comparable scale (see the
# "Scaling and weighting criteria" tutorial); we sum-scale both the
# matrix and the weights, exactly as in the original study.
dms = SumScaler(target="both").transform(dm)
dms

,C1[▲ 0.25],C2[▲ 0.25],C3[▼ 0.10],C4[▲ 0.20],C5[▲ 0.20]
L1,0.18750,0.172414,0.163743,0.121951,0.113636
L2,0.12500,0.068966,0.146199,0.243902,0.204545
L3,0.15625,0.241379,0.204678,0.219512,0.136364
L4,0.18750,0.034483,0.157895,0.146341,0.159091
L5,0.18750,0.275862,0.175439,0.170732,0.204545
L6,0.15625,0.206897,0.152047,0.097561,0.181818


## 1. Recap: how `ELECTRE1` builds its kernel

`ELECTRE1` compares every pair of alternatives $(a, b)$ using two
indices:

- **Concordance** $C(a,b)$: the (weighted) proportion of criteria where
  $a$ is at least as good as $b$ — "how much evidence supports
  $a \succeq b$".
- **Discordance** $D(a,b)$: how large the *worst* disagreement is — the
  biggest normalized gap on any single criterion where $b$ beats $a$ —
  "how strong the veto against $a \succeq b$ is".

$a$ **outranks** $b$ only if the evidence is strong enough
($C(a,b) \geq p$) *and* no single criterion vetoes it
($D(a,b) \leq q$). The two low-level functions that compute these
matrices are available directly:

In [2]:
import pandas as pd
from skcriteria.agg.electre import concordance, discordance

alts = dm.alternatives
c_matrix = concordance(dms.matrix.to_numpy(), dms.iobjectives.to_numpy(), dms.weights.to_numpy())
d_matrix = discordance(dms.matrix.to_numpy(), dms.iobjectives.to_numpy())

display(pd.DataFrame(c_matrix, index=alts, columns=alts).round(3))
display(pd.DataFrame(d_matrix, index=alts, columns=alts).round(3))

,L1,L2,L3,L4,L5,L6
L1,NaN,0.50,0.35,0.50,0.35,0.45
L2,0.50,NaN,0.50,0.75,0.50,0.50
L3,0.65,0.50,NaN,0.45,0.20,0.70
L4,0.75,0.25,0.55,NaN,0.35,0.45
L5,0.90,0.70,0.80,0.90,NaN,0.90
L6,0.55,0.50,0.55,0.55,0.10,NaN


,L1,L2,L3,L4,L5,L6
L1,NaN,0.505,0.404,0.188,0.429,0.282
L2,0.429,NaN,0.714,0.259,0.857,0.571
L3,0.170,0.282,NaN,0.194,0.282,0.218
L4,0.571,0.404,0.857,NaN,1.000,0.714
L5,0.048,0.303,0.202,0.073,NaN,0.097
L6,0.129,0.606,0.505,0.202,0.303,NaN


With the default thresholds ($p=0.65$, $q=0.35$), `ELECTRE1` keeps only
the alternatives that no other alternative outranks — the **kernel**:

In [3]:
from skcriteria.agg.electre import ELECTRE1

e1 = ELECTRE1().evaluate(dms)
display(e1)
print("Kernel alternatives:", list(e1.kernel_alternatives_))

Alternatives,L1,L2,L3,L4,L5,L6
Kernel,False,False,False,False,True,False


Kernel alternatives: ['L5']


Only `L5` survives. Looking back at the concordance/discordance tables,
`L5` has the highest concordance (and lowest discordance) against every
other alternative — it is decisively good almost everywhere.

## 2. `ELECTRE2`: from a kernel to a full ranking

`ELECTRE1` only tells us *which* alternatives are good enough to keep —
it says nothing about how the other five compare to each other.
`ELECTRE2` <cite data-cite="roy1971methode">[roy1971methode]</cite>
<cite data-cite="roy1973methode">[roy1973methode]</cite> fixes this by
introducing **two levels of outranking** instead of one:

- **Strong outranking**: $a$ clearly beats $b$ (high concordance *and*
  low discordance, using the strictest thresholds `p0`/`q0`, or a
  slightly relaxed pair `p1`/`q1`).
- **Weak outranking**: $a$ beats $b$, but less decisively (thresholds
  `p2`/`q0`).

Combining both relations lets `ELECTRE2` build a *complete* ranking
instead of a kernel:

In [4]:
from skcriteria.agg.electre import ELECTRE2

e2 = ELECTRE2().evaluate(dms)
e2

Alternatives,L1,L2,L3,L4,L5,L6
Rank,6.000000,3.000000,2.000000,5.000000,1.000000,4.000000


`L5` is still the winner (consistent with being the sole `ELECTRE1`
kernel member), but now every alternative has a place — `L3` is second,
`L2` third, and so on. This comes at a price: five parameters instead of
two (`p0`, `p1`, `p2`, `q0`, `q1`), each with a specific role (see the
class docstring for the exact strong/weak outranking rules).

## 3. Reading the output: kernel vs. total order

It is tempting to treat `ELECTRE1`'s kernel as "the top of the ranking",
but that is not quite right: the kernel has **no internal order**, and
alternatives *outside* the kernel are not necessarily ordered among
themselves either — some pairs are simply **incomparable**. We can see
this directly in `ELECTRE1`'s outranking matrix (available as
`e1.e_.outrank`, `outrank[i, j]` is `True` when alternative `i`
outranks `j`):

In [5]:
outrank = e1.e_.outrank
pd.DataFrame(outrank, index=alts, columns=alts)

,L1,L2,L3,L4,L5,L6
L1,False,False,False,False,False,False
L2,False,False,False,True,False,False
L3,True,False,False,False,False,True
L4,False,False,False,False,False,False
L5,True,True,True,True,False,True
L6,False,False,False,False,False,False


`L5` outranks every other alternative (its row is all `True`), and
nothing outranks `L5` (its column is all `False`) — that is exactly why
it is the only kernel member. But look at `L2` and `L3`: neither
outranks the other, and neither is in the kernel. `ELECTRE1` simply has
no opinion about which of the two is better; asking it to rank them
would be over-interpreting the result. `ELECTRE2`, which *does* commit
to a total order, resolves this ambiguity — but only because it
introduces the extra weak-outranking machinery from Part 2 to break
ties that `ELECTRE1` deliberately leaves open.

<div class="alert alert-warning">

**Rule of thumb:** use `ELECTRE1` when you only need to shortlist a
handful of solid candidates (and are fine leaving them unordered); use
`ELECTRE2` when a decision-maker needs a single, total order to act on.

</div>

## 4. Threshold sensitivity

`p`/`q` (and `ELECTRE2`'s five thresholds) are not just tuning knobs —
they directly control how *strict* the outranking relation is, and
therefore how large the kernel ends up being. Stricter thresholds (high
`p`, low `q`) demand more decisive evidence, so fewer alternatives
survive; looser thresholds let more alternatives into the kernel.

We reproduce the sensitivity experiment from
<cite data-cite="barba1997decisiones">[barba1997decisiones]</cite>: nine
increasingly permissive `(p, q)` pairs applied to a six-alternative
matrix.

In [6]:
sensitivity_dm = skc.mkdm(
    matrix=[
        [0.188, 0.172, 0.168, 0.122, 0.114],
        [0.125, 0.069, 0.188, 0.244, 0.205],
        [0.156, 0.241, 0.134, 0.220, 0.136],
        [0.188, 0.034, 0.174, 0.146, 0.159],
        [0.188, 0.276, 0.156, 0.171, 0.205],
        [0.156, 0.207, 0.180, 0.098, 0.182],
    ],
    weights=[0.25, 0.25, 0.10, 0.20, 0.20],
    objectives=[max, max, max, max, max],
    alternatives=["A", "B", "D", "E", "G", "H"],
)
sensitivity_dms = SumScaler(target="both").transform(sensitivity_dm)

ps = [0.50, 0.60, 0.70, 0.80, 0.89, 0.89, 0.89, 0.94, 1.00]
qs = [0.50, 0.40, 0.30, 0.20, 0.10, 0.08, 0.05, 0.05, 0.00]

rows = []
for p, q in zip(ps, qs):
    kernel = ELECTRE1(p=p, q=q).evaluate(sensitivity_dms).kernel_alternatives_
    rows.append({"p": p, "q": q, "kernel_size": len(kernel), "kernel": ", ".join(kernel)})

pd.DataFrame(rows)

,p,q,kernel_size,kernel
0,0.50,0.50,1,G
1,0.60,0.40,1,G
2,0.70,0.30,2,"B, G"
3,0.80,0.20,3,"B, D, G"
4,0.89,0.10,3,"B, D, G"
5,0.89,0.08,4,"B, D, G, H"
6,0.89,0.05,5,"B, D, E, G, H"
7,0.94,0.05,6,"A, B, D, E, G, H"
8,1.00,0.00,6,"A, B, D, E, G, H"


As the thresholds relax (`p` grows, `q` shrinks), the kernel grows
**monotonically** from a single alternative (`G`, the most consistently
strong one — always present) to all six once the thresholds are
essentially permissive enough to accept everyone. In practice, `p` and
`q` should be chosen (or swept, as above) together with the
decision-maker, since they directly encode *how much* evidence and
*how large* a single-criterion disadvantage are tolerated before ruling
an alternative out.

## 5. When to choose ELECTRE over TOPSIS/WeightedSumModel

`TOPSIS` and `WeightedSumModel` are **compensatory**: a very high score
on one criterion can offset a very low score on another, because both
end up added into a single number. `ELECTRE`'s concordance/discordance
machinery is only *partially* compensatory — the discordance/veto check
means that a large enough single-criterion disadvantage can block an
outranking relation regardless of how good the alternative is elsewhere.

To see the difference concretely, we use the seven-criteria matrix from
<cite data-cite="wang2006ranking">[wang2006ranking]</cite> (all
`Maximize`, so no inversion is needed) and compare `ELECTRE2` against
`WeightedSumModel` and `TOPSIS`.

In [7]:
wang_dm = skc.mkdm(
    matrix=[
        [1, 2, 1, 5, 2, 2, 4],
        [3, 5, 3, 5, 3, 3, 3],
        [3, 5, 3, 5, 3, 2, 2],
        [1, 2, 2, 5, 1, 1, 1],
        [1, 1, 3, 5, 4, 1, 5],
    ],
    objectives=[max, max, max, max, max, max, max],
    weights=[0.0780, 0.1180, 0.1570, 0.3140, 0.2350, 0.0390, 0.0590],
    alternatives=["A1", "A2", "A3", "A4", "A5"],
)

from skcriteria.agg.simple import WeightedSumModel
from skcriteria.agg.topsis import TOPSIS
from skcriteria.cmp import mkrank_cmp

electre2_rank = ELECTRE2().evaluate(wang_dm)
wsm_rank = WeightedSumModel().evaluate(wang_dm)
topsis_rank = TOPSIS().evaluate(wang_dm)

rcmp = mkrank_cmp(electre2_rank, wsm_rank, topsis_rank)
rcmp.to_dataframe()

Method,ELECTRE2,WeightedSumModel,TOPSIS
Alternatives,,,
A1,4.0,4,4
A2,1.0,1,1
A3,3.0,2,2
A4,5.0,5,5
A5,2.0,3,3


`WeightedSumModel` and `TOPSIS` agree on every alternative, but
`ELECTRE2` swaps `A3` and `A5`. Looking at the raw data explains why:
`A5` is much weaker than `A3` on `C1` and `C2` (its two lowest scores in
the whole matrix, `1` vs. `A3`'s `3` and `5`), but the loss is more than
compensated, *in a purely additive sense*, by `A5` being noticeably
better on `C5` and `C7`. `WeightedSumModel`/`TOPSIS` reward that trade;
`ELECTRE2`'s pairwise concordance/discordance comparison weighs the
*consistency* of the advantage differently and reaches the opposite
conclusion for this particular pair.

Neither answer is "more correct" in the abstract — it depends on
whether the decision-maker considers a large deficit on one criterion
acceptable as long as others compensate. As a rule of thumb, prefer
`ELECTRE1`/`ELECTRE2` over `TOPSIS`/`WeightedSumModel`/`WASPAS` (and the
other methods from the
[classic aggregation methods](agg_methods.ipynb) tutorial) when:

- Criteria are **not meant to be compensatory** (e.g. a safety or legal
  requirement should not be "bought back" by a high score elsewhere).
- You only need a **shortlist**, not a full ranking (`ELECTRE1`).
- Criteria are measured on genuinely different, hard-to-normalize
  scales, since concordance/discordance only need an ordinal comparison
  per criterion rather than a fully commensurable score.